# 02 · Modello — NeuroLoc-Δ v2: Double-Head, card, struttura, calibrazione, distillazione

**Cosa predice.** Per ogni proteina umana del cervello:
- **Asse A — 9 organelli:** una *distribuzione* (Softmax): "quanto sta dove", in %, somma 100.
- **Asse B — 5 domini neuronali** (assone, dendrite, presinapsi, postsinapsi, mielina): cinque *probabilità indipendenti* (Sigmoide). Non si sommano: un mitocondrio può stare in un assone.

## Schema del ragionamento
```
 SEQUENZA ESM-2 650M ──┬─ intera (media su finestre sovrapposte)           1280 ─┐
                       └─ 3 frammenti: N-term 100 · centro ±100 · C-term 100  3×1280 ┤  4 token chiave/valore
 CARD (BioLinkBERT-large, 1024) ─ [gate: poche parole → conta poco] ─ [card-dropout 35 %] ─ query ──┤  CROSS-ATTENTION: la card sceglie
 STRUTTURA: AlphaFold (pLDDT, disordine, esposizione terminali, eliche TM, toppa K)                │  quale pezzo di proteina interrogare
          + descrittori di sorting (NLS, KDEL, PTS1, PDZ, CAAX…) + espressione cerebrale + af_missing ┘
                                              │
                                        fusione → repr_256 ──┬─ head_org (9)  Softmax → % per organello   [KL + λ·focale]
                                              ▲              └─ head_neu (5)  Sigmoide → P per dominio    [Asymmetric Loss + pesi class-balanced]
                                   SupCon sulle classi rare       peso dei due task: incertezza omoscedastica (Kendall)
 Poi: Temperature Scaling · Student (solo sequenza+AlphaFold) distillato dal Teacher per le proteine orfane
```

## Analisi esperta del piano di miglioramento: cosa è stato accolto, corretto o respinto
| Punto | Esito | Perché |
|---|---|---|
| **Double-Head** (Softmax A + Sigmoide B) | ✅ accolto | Gli organelli *competono* per la massa proteica (partizione), i domini neuronali no: una sola Softmax li costringeva a sommare 1 |
| **Bug 2 · Kendall al posto di α fisso** | ✅ accolto, con limite | la forma `exp(−s)·L + s` porta il peso a ≈ 1/L: un task con loss piccola prende peso enorme → `s` limitato a [−3, 3]; la "KL" del piano è in realtà cross-entropy (offset = entropia) → KL vera |
| **Asymmetric Loss** + pesi `W_j = (1−β)/(1−β^N_j)` | ⚠️ accolto con ri-scalatura | `W_j` grezzo vale ~0,25 per N=4 e ~0,002 per N=1000: moltiplicato ai soli positivi li rimpicciolisce ~100× rispetto ai negativi. Mediana → 1, ritaglio [0,5; 10] |
| **Bug 1 · fold** `MultilabelStratifiedKFold` | ⚠️ accolto in forma migliore | ignora l'omologia (paraloghi in fold diversi = leakage). Qui stratificazione iterativa **su gruppi di omologia**; con N<5 positivi nessun fold può averne uno → metriche sulle OOF *riunite* |
| **Bug 1 · silver-standard mielina** | ⚠️ corretto (notebook 01) | `is_oligo ∧ brain_ntpm≥20` = migliaia di proteine domestiche. Criterio di arricchimento ≥4×, peso 0,6, mai nelle metriche |
| **Bug 5 · `af_missing`** | ✅ accolto | NaN→0 *dopo* la standardizzazione + flag; il piano lo definiva come "tutte le feature a zero", ambiguo |
| **Card-Dropout 0,35 + vettore nullo** | ✅ accolto | |
| **Dynamic gate** | ⚠️ corretto | nel piano `gate_w` è usato solo in inferenza → resta 1.0 per sempre (nessun gradiente) e con n=0 il gate vale 0,5. Qui lavora anche in training, scala logaritmica, con bias |
| **"Cross-attention" Q=testo, K/V=sequenza** | ❌ difetto nel piano | una query e una chiave: il softmax su 1 elemento = 1, l'output non dipende dalla query (è una mappa lineare). Qui 4 token K/V → attenzione vera |
| **SupCon** | ⚠️ accolto + 2 correzioni | media solo sugli anchor con almeno un positivo; campionatore che sovra-rappresenta i positivi rari, altrimenti nel batch non ci sono coppie |
| **ST3 · `card_degraded`, `I_leak`** | ✅ / ⚠️ | implementati. `I_leak` misura informazione *residua*, non leakage: la funzione è informativa legittimamente. La soglia 0,35 è arbitraria: si stampa, ma la prova è l'ablazione |
| **ST4 · Card-Free, Low-Annotation** | ✅ con correzione | la soglia di 15 parole va sulla colonna `card_n_annot` (vedi 01) |
| **ST5 · Temperature Scaling** | ✅ | anche per le sigmoidi (T_neu); `frac_cyto` è la quota *solubile*, non identica alla quota "Cytoplasm" |
| **ST6 · pesi stratificati** | ✅ | LOPIT/OpenCell/Hein/Itzhak (linee tumorali) pesano 0 per i domini neuronali |
| **ST6b · KD** | ⚠️ accolto + λ·loss dura | un Student solo-distillazione eredita gli errori del Teacher, sovra-sicuro sul proprio training |
| **Affermazione "calo Full→Card-Free < 10 %"** | ➖ trasformata in test | non si può promettere: si addestra il gemello senza dropout e si stampa RISPETTATO / NON rispettato |

## Domande da esperto, con risposta
1. **Perché la card non deve contenere la localizzazione?** Perché UniProt la scrive nel testo della funzione: il modello leggerebbe la risposta. → 3 versioni (`full` ≥ `protein` ≥ `degraded`) e ablazione.
2. **Allora il testo è inutile?** No: funzione, malattia, partner ed espressione sono informazione biologica vera. Ma per una proteina *orfana* la card è vuota → Student + Card-Dropout.
3. **Perché i frammenti N/centro/C separati?** Il segnale di sorting è locale (peptide di transito mitocondriale, KDEL, PTS1…) e la media su 1000 residui lo diluisce.
4. **Perché la mielina non si valuta come le altre?** Pochi positivi reali: le metriche sono marcate non affidabili sotto 10 positivi, e le predizioni di mielina si leggono come ipotesi.
5. **Perché i pesi delle fonti sono diversi per i due assi?** Un'immagine di HeLa non dice nulla su un assone.
6. **Perché fold per omologia?** CHRNA1 in train e CHRNA4 in test gonfierebbero tutte le metriche.

## Letteratura da cui si "ruba" il ragionamento
| Lavoro | Idea usata | Dove |
|---|---|---|
| DeepLoc 2.0 (Thumuluri 2022) | embedding di language model + attenzione, test indipendente HPA, soglie per MCC | baseline "DeepLoc-style", test HPA, MCC; attention-pooling in 03 |
| ProtST (Xu 2023) | allineare sequenza e testo di funzione | InfoNCE in 03 |
| ESM-2 (Lin 2023) · BioLinkBERT (Yasunaga 2022) | encoder di sequenza e di testo biomedico | §Embedding |
| AlphaFold DB (Jumper 2021; Varadi 2022) | pLDDT come disordine, esposizione dei terminali | `af_*` |
| Asymmetric Loss (Ben-Baruch 2020) · Class-Balanced (Cui 2019) | classi rare multi-label | head_neu |
| Kendall 2018 · SupCon (Khosla 2020) · Guo 2017 · Hinton 2015 | pesi multi-task, geometria, calibrazione, distillazione | §loss, §ST5, §ST6 |
| SynGO (Koopmans 2019) · Hein 2024 · Itzhak 2016 · HPA | etichette neuronali e quote misurate | notebook 01 |

## Limiti dichiarati
Nessun risultato numerico è riportato qui: li produce l'esecuzione. Le quote misurate vengono da linee cellulari, non da neuroni. Le classi neuronali hanno etichette *a mondo aperto* (l'assenza di annotazione non è assenza di proteina). Il raggruppamento per omologia usa MMseqs2 se installato, altrimenti famiglia UniProt + cluster ESM (più approssimato).

## Roadmap aggiornata
✅ dati cervello-umano · ✅ card in 3 versioni · ✅ Double-Head · ✅ calibrazione · ✅ Student ·
⏭ cluster MMseqs2 al 30 % · ⏭ mappe assonali/dendritiche iPSC come fonti forti · ⏭ stadio a grafo (partner di complesso) · ⏭ varianti patogene (mislocalizzazione) · ⏭ validazione su un test neuronale esterno (script 03 e oltre)

## Configurazione
Iperparametri in `HYPER`. `RUN_ABLATIONS=False` salta le 4 CV di ablazione (~⅔ del tempo).

In [ ]:
import copy, gc, json, math, re, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score, roc_curve
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA, EMB, MODELS, RESULTS, FIG = Path('data'), Path('data/embeddings'), Path('models'), Path('results'), Path('figures')
for d in [EMB, MODELS, RESULTS, FIG]:
    d.mkdir(parents=True, exist_ok=True)

ESM_NAME = 'facebook/esm2_t33_650M_UR50D'          # ESM-2 650M (1280-d): scelta fissa del piano
BERT_NAME = 'michiyasunaga/BioLinkBERT-large'       # 1024-d
N_WIN, CENTER_HALF = 100, 100                       # frammenti: N-term 100 aa · centro = metà ± 100 aa (200 aa) · C-term 100 aa
MAXRES, STRIDE, TOK_BUDGET = 1022, 767, 6000        # ESM-2 legge ≤ 1022 residui: oltre, finestre sovrapposte e media

HYPER = dict(proj_dim=512, n_heads=8, dropout=0.3, card_drop_prob=0.35,
             lr=3e-4, lr_logvar=1e-3, wd=1e-2, batch=128, epochs=60, patience=8,
             kl_gamma=2.0, kl_lambda=0.1,                         # Asse A: KL + λ·regolarizzatore focale
             asl_gneg=4, asl_gpos=1, asl_clip=0.05, cb_beta=0.999,   # Asse B: Asymmetric Loss + pesi class-balanced
             beta_supcon=0.1, supcon_rare_max=500, supcon_temp=0.07,
             T_kd=4.0, beta_feat=0.01, lam_hard=0.5)              # distillazione
N_FOLDS, MIN_EVAL_POS, LOW_ANNOT_WORDS = 5, 10, 15
RUN_ABLATIONS = True            # card full/degraded, senza card-dropout, senza SupCon: ~6 CV complete
KD_USE_UNLABELED = True         # lo Student impara dal Teacher anche su proteine del cervello senza etichetta (mai quelle del test)
print('device:', DEVICE)

## Dati dal notebook 01

In [ ]:
prot = pd.read_parquet(DATA / 'proteine.parquet').sort_values('pid').reset_index(drop=True)
obs = pd.read_parquet(DATA / 'osservazioni.parquet')
cfg_dati = json.load(open(DATA / 'config_dati.json'))
ALL, ORG, NEU = cfg_dati['ALL'], cfg_dati['ORGANELLES'], cfg_dati['NEURO']
N, NO, NN = len(prot), len(ORG), len(NEU)
assert (prot['pid'].to_numpy() == np.arange(N)).all()
is_train = (prot['role'] == 'train').to_numpy()
is_test = (prot['role'] == 'test').to_numpy()
seqs = prot['sequence'].tolist()
print(f'{N:,} proteine · addestrabili {int(prot.has_label.sum()):,} · test HPA {int(is_test.sum()):,}')
print(f'Asse A (softmax, 100 %): {ORG}\nAsse B (sigmoidi indipendenti): {NEU}')

## Embedding ESM-2 650M: sequenza intera + frammenti (N 100 · centro ±100 · C 100)
La finestra centrale è volutamente più larga (200 aa): i segnali terminali sono corti, il centro porta contesto di dominio. Le proteine > 1022 aa (SHANK3, BSN, PCLO, ANK3) non si troncano: finestre sovrapposte e media.

In [ ]:
# ---------- ESM-2 650M: embedding della sequenza intera + 3 frammenti (N-term 100, centro ±100, C-term 100) -------------------------
# E_seq  (N, 1280)  media dell'ultimo strato su TUTTA la proteina (finestre sovrapposte oltre 1022 aa: SHANK, BSN, PCLO, ANK3… non si troncano)
# E_frag (N, 3840)  i tre frammenti embeddati ciascuno da solo e concatenati [N | centro | C]
ESM_DIM = 1280

def fragments(s):
    mid = len(s) // 2
    return s[:N_WIN], s[max(0, mid - CENTER_HALF):mid + CENTER_HALF], s[-N_WIN:]

def open_mm(path, dtype, shape):
    return np.lib.format.open_memmap(path, mode='r+' if path.exists() else 'w+', dtype=dtype, shape=shape)

E_seq = open_mm(EMB / 'esm_seq.npy', np.float16, (N, ESM_DIM))
E_frag = open_mm(EMB / 'esm_frag.npy', np.float16, (N, 3 * ESM_DIM))
DONE = open_mm(EMB / 'esm_done.npy', np.bool_, (N,))

if not DONE[:].all():
    from transformers import AutoTokenizer, EsmModel
    tok_esm = AutoTokenizer.from_pretrained(ESM_NAME)
    esm = EsmModel.from_pretrained(ESM_NAME, add_pooling_layer=False).to(DEVICE).eval()
    if DEVICE == 'cuda':
        esm = esm.half()

    @torch.inference_mode()
    def esm_hidden(batch):
        enc = tok_esm(batch, return_tensors='pt', padding=True).to(DEVICE)
        h = esm(**enc).last_hidden_state[:, 1:].float()          # via il token BOS: la posizione 0 è il primo residuo
        return h, enc['attention_mask'].sum(1) - 2                # lunghezze in residui (senza BOS/EOS)

    def mean_pool(h, lens):
        m = (torch.arange(h.shape[1], device=h.device)[None] < lens[:, None]).unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    def frag_embed(idx):
        parts = [fragments(seqs[i]) for i in idx]
        out = []
        for k in range(3):
            h, ln = esm_hidden([p[k] for p in parts])
            out.append(mean_pool(h, ln))
        return torch.cat(out, 1).cpu().numpy().astype(np.float16)

    L = np.array([len(s) for s in seqs])
    todo = np.flatnonzero(~DONE[:])
    short = todo[L[todo] <= MAXRES]
    order = short[np.argsort(L[short])]
    t0, i, nb = time.time(), 0, 0
    while i < len(order):
        j = i
        while j < len(order) and (j - i + 1) * (L[order[j]] + 2) <= TOK_BUDGET:
            j += 1
        j = max(j, i + 1)
        idx, i = order[i:j], j
        h, ln = esm_hidden([seqs[k] for k in idx])
        E_seq[idx] = mean_pool(h, ln).cpu().numpy().astype(np.float16)
        E_frag[idx] = frag_embed(idx)
        DONE[idx] = True
        nb += 1
        if nb % 20 == 0:
            DONE.flush(); E_seq.flush(); E_frag.flush()
            print(f'ESM-2: {i:,}/{len(order):,} corte ({time.time() - t0:.0f}s)')
    for i in todo[L[todo] > MAXRES]:                              # proteine lunghe: finestre sovrapposte, media per residuo
        s = seqs[i]
        starts = sorted(set(list(range(0, len(s) - MAXRES, STRIDE)) + [len(s) - MAXRES]))
        acc, cnt = torch.zeros(len(s), ESM_DIM), torch.zeros(len(s), 1)
        for st in starts:
            h, ln = esm_hidden([s[st:st + MAXRES]])
            n = int(ln[0])
            acc[st:st + n] += h[0, :n].cpu(); cnt[st:st + n] += 1
        E_seq[i] = (acc / cnt).mean(0).numpy().astype(np.float16)
        E_frag[i] = frag_embed([i])[0]
        DONE[i] = True
    DONE.flush(); E_seq.flush(); E_frag.flush()
    del esm; gc.collect(); torch.cuda.empty_cache()
E_seq, E_frag = np.array(E_seq), np.array(E_frag)
print('embedding ESM-2:', E_seq.shape, E_frag.shape)

## Embedding della card (BioLinkBERT-large, 1024-d) in tre versioni

In [ ]:
# ---------- BioLinkBERT-large (1024-d): embedding della card in TRE versioni (protein · full · degraded) ---------------------------
# card lunghe → blocchi da ~280 parole (≤ 512 token), embedding medio dei blocchi.
BERT_DIM = 1024
CARD_MODES = ['protein', 'full', 'degraded']
col_of = {'protein': 'card_protein', 'full': 'card_full', 'degraded': 'card_degraded'}
E_text = {}
if not all((EMB / f'bert_{m}.npy').exists() for m in CARD_MODES):
    from transformers import AutoModel, AutoTokenizer
    tok_b = AutoTokenizer.from_pretrained(BERT_NAME)
    bert = AutoModel.from_pretrained(BERT_NAME).to(DEVICE).eval()
    if DEVICE == 'cuda':
        bert = bert.half()

    @torch.inference_mode()
    def embed_texts(texts, bs=32):
        chunks, owner = [], []
        for i, t in enumerate(texts):
            w = str(t).split()
            for j in range(0, max(len(w), 1), 280):
                chunks.append(' '.join(w[j:j + 280])); owner.append(i)
        order = np.argsort([len(c) for c in chunks])
        out = np.zeros((len(chunks), BERT_DIM), np.float32)
        for s in range(0, len(order), bs):
            ids = order[s:s + bs]
            enc = tok_b([chunks[k] for k in ids], padding=True, truncation=True, max_length=512, return_tensors='pt').to(DEVICE)
            h = bert(**enc).last_hidden_state.float()
            m = enc['attention_mask'].unsqueeze(-1).float()
            out[ids] = F.normalize((h * m).sum(1) / m.sum(1), dim=-1).cpu().numpy()
        owner = np.array(owner)
        res = np.zeros((len(texts), BERT_DIM), np.float32)
        np.add.at(res, owner, out)
        return (res / np.bincount(owner, minlength=len(texts))[:, None]).astype(np.float16)

    for m in CARD_MODES:
        if not (EMB / f'bert_{m}.npy').exists():
            np.save(EMB / f'bert_{m}.npy', embed_texts(prot[col_of[m]].tolist()))
            print('card embeddata:', m)
    del bert; gc.collect(); torch.cuda.empty_cache()
for m in CARD_MODES:
    E_text[m] = np.load(EMB / f'bert_{m}.npy')
print({m: v.shape for m, v in E_text.items()})

## Feature: descrittori di sorting + AlphaFold + espressione, con flag `af_missing`
`ctx_n_localizzazioni_hpa` e `ctx_shuttle_hpa` (presenti nel parquet) sono **esclusi**: derivano dalla localizzazione HPA, cioè dalla risposta. Si stampa `STRUCT_DIM` (richiesto da ST1).

In [ ]:
# ---------- F_struct = descrittori di sorting dalla sequenza + AlphaFold + espressione cerebrale (+ flag af_missing, Bug 5) --------------
# Il language model vede il contesto; questi numeri codificano segnali noti di smistamento con una forma esplicita: NLS/NES, KDEL, PTS1, CAAX,
# ligandi PDZ, dileucina/YxxΦ, eliche idrofobiche (TM), peptide segnale, MTS anfipatico, ripetizioni KXGS (tau/MAP), poliQ, RG.
AAS = 'ACDEFGHIKLMNPQRSTVWY'
KD = dict(zip(AAS, [1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3]))
HYD = np.array([KD[a] for a in AAS])
RX = {k: re.compile(v) for k, v in dict(
    kdel=r'[KHRQSAE][DE]EL$', pts1=r'[SAC][KRH][LMI]$', pdz=r'[ST].[VILF]$', caax=r'C[ALIVM][ALIVM]\w$', nls=r'[KR][KR].[KR]|K[KR].[KR]|P.{0,2}K[KR]..?[KR]',
    nes=r'L.{2,3}[LIVFM].{2,3}L.[LI]', dileu=r'[DE].{2,3}L[LI]', yxxphi=r'Y..[LIVMF]', pxxp=r'P..P', kxgs=r'K.GS', rg=r'RG{1,2}', polyq=r'Q{7,}',
    polyp=r'P{5,}', walker=r'G.{4}GK[ST]', ef=r'D.[DNS].[DNS]G.I..{3}E').items()}
HAND_COLS = (['len_log'] + ['aa_' + a for a in AAS] +
             ['kd_mean', 'kd_win19_max', 'n_tm_win', 'sp_hphob_max30', 'sp_nregion_pos', 'sp_start_M', 'mts_charge_pos', 'mts_hphob', 'mts_arg_ser',
              'nt_charge', 'ct_charge', 'ct_hphob', 'net_charge_ph', 'frac_disorder_aa', 'frac_acidic', 'frac_basic', 'cys_nt15', 'cys_total_frac',
              'ct4_kdel', 'ct3_pts1', 'ct3_pdz', 'ct4_caax', 'n_nls', 'n_nes', 'n_dileu', 'n_yxxphi', 'n_pxxp', 'n_kxgs', 'n_rg', 'polyq', 'polyp',
              'walker', 'ef_hand', 'basic_cluster_max', 'acid_cluster_max'])

def hand_features(s):
    L = len(s)
    kd = np.array([KD.get(a, 0.0) for a in s])
    comp = np.array([s.count(a) for a in AAS]) / L
    win = np.convolve(kd, np.ones(19) / 19, mode='valid') if L >= 19 else np.array([kd.mean()])
    n30, c30 = s[:30], s[-30:]
    pos = lambda t: sum(t.count(a) for a in 'KR') - sum(t.count(a) for a in 'DE')
    sp = np.array([KD.get(a, 0.0) for a in n30[1:25]])
    clus = lambda letters: max([sum(a in letters for a in s[i:i + 8]) for i in range(0, max(1, L - 7), 4)] or [0])
    f = [math.log(L)] + list(comp) + [
        kd.mean(), win.max(), int((win > 1.6).sum() > 0) + int(((win[1:] > 1.6) & (win[:-1] <= 1.6)).sum()),
        sp.max() if len(sp) else 0.0, pos(s[1:12]), float(s[0] == 'M'), pos(n30[:25]),
        np.mean([KD.get(a, 0.0) for a in n30[:25]]), (n30[:25].count('R') + n30[:25].count('S')) / max(len(n30[:25]), 1),
        pos(n30), pos(c30), np.mean([KD.get(a, 0.0) for a in c30]), pos(s) / L,
        sum(comp[AAS.index(a)] for a in 'GPSQEKNDR'), comp[AAS.index('D')] + comp[AAS.index('E')], comp[AAS.index('K')] + comp[AAS.index('R')],
        s[:15].count('C'), s.count('C') / L,
        float(bool(RX['kdel'].search(s))), float(bool(RX['pts1'].search(s))), float(bool(RX['pdz'].search(s))), float(bool(RX['caax'].search(s))),
        len(RX['nls'].findall(s)), len(RX['nes'].findall(s)), len(RX['dileu'].findall(s)), len(RX['yxxphi'].findall(s)), len(RX['pxxp'].findall(s)),
        len(RX['kxgs'].findall(s)), len(RX['rg'].findall(s)), float(bool(RX['polyq'].search(s))), float(bool(RX['polyp'].search(s))),
        float(bool(RX['walker'].search(s))), float(bool(RX['ef'].search(s))), clus('KR'), clus('DE')]
    return f

hand = np.array([hand_features(s) for s in seqs], dtype=np.float32)
assert hand.shape[1] == len(HAND_COLS), (hand.shape, len(HAND_COLS))

AF_COLS = [c for c in prot.columns if c.startswith('af_') and c != 'af_missing']
# espressione cerebrale e contesto SENZA etichette: ctx_n_localizzazioni_hpa e ctx_shuttle_hpa derivano dalla localizzazione HPA → sono la risposta, esclusi.
CTX_COLS = ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm', 'is_neuronal', 'is_oligo', 'ctx_struttura_pdb', 'ctx_ruolo_trafficking']
ctx_m = prot[CTX_COLS].astype(float).copy()
for c in ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm']:
    ctx_m[c] = np.log1p(ctx_m[c].fillna(0))
af_missing = prot['af_missing'].fillna(1.0).to_numpy(np.float32)[:, None] if 'af_missing' in prot else np.ones((N, 1), np.float32)
F_raw = np.hstack([prot[AF_COLS].astype(float).to_numpy(), hand, ctx_m.to_numpy()]).astype(np.float32)
mu = np.nanmean(F_raw[is_train], 0); sd = np.nanstd(F_raw[is_train], 0) + 1e-6      # statistiche non supervisionate, solo proteine "train"
F_struct = np.nan_to_num((F_raw - mu) / sd, nan=0.0).clip(-6, 6).astype(np.float32)       # NaN → 0 DOPO la standardizzazione; il flag dice che era un NaN
STRUCT_DIM = F_struct.shape[1]
card_n_annot = prot['card_n_annot'].to_numpy(np.float32)
print(f'STRUCT_DIM={STRUCT_DIM}  (AlphaFold {len(AF_COLS)} + sequenza {hand.shape[1]} + espressione {len(CTX_COLS)}); '
      f'proj_struct riceve STRUCT_DIM+1={STRUCT_DIM + 1} con il flag af_missing · senza struttura: {int(af_missing.sum())} proteine')

## Etichette: soft label (Asse A e B), pesi per fonte stratificati per asse, riferimento REF e bilanciamento
**Come si legge:** per l'Asse A i voti pesati dei positivi diventano una distribuzione (due fonti dicono Citoplasma, una Nucleo → 0,67/0,33). Per l'Asse B ogni elemento vale il peso della migliore fonte positiva; gli zeri vengono solo dalle fonti con peso > 0 e solo per proteine neuronali (mielina: oligodendrocitarie). **REF** (solo fonti forti, silver escluso) è il riferimento binario con cui si calcolano le metriche.

In [ ]:
# ---------- Etichette: soft label per l'Asse A, soft multi-label per l'Asse B, riferimento binario (REF) per le metriche ------------------
# PESI STRATIFICATI (piano, Sub-Task 6): i dati di LOPIT (U2OS/HeLa), OpenCell e Hein/Itzhak (HEK293T/HeLa) vengono da LINEE TUMORALI. Per
# citosol, nucleo, mitocondrio… (organelli ubiquitari) sono validi; non hanno però misurato assone, dendrite, sinapsi o mielina, quindi per l'Asse B
# pesano 0. Per i domini neuronali valgono solo le fonti con evidenza neuronale: SynGO (curata a mano), GOA sperimentale, UniProt forte, mappe iPSC.
PESI_ORG = {'deeploc': 1.0, 'mitocarta': 1.0, 'uniprot_forte': 0.9, 'hpa_alta': 0.8, 'goa_exp': 0.75, 'opencell': 0.6,
            'lopit_hyper': 0.5, 'lopit_dc': 0.5, 'lopit_2024': 0.5, 'bioid': 0.5, 'hpa_approved': 0.4, 'uniprot_debole': 0.3}
PESI_NEU = {'syngo': 1.0, 'goa_exp': 0.85, 'uniprot_forte': 0.7, 'silver_oligo': 0.6, 'uniprot_debole': 0.2,
            'deeploc': 0.0, 'mitocarta': 0.0, 'hpa_alta': 0.0, 'hpa_approved': 0.0, 'opencell': 0.0,
            'lopit_hyper': 0.0, 'lopit_dc': 0.0, 'lopit_2024': 0.0, 'bioid': 0.0, 'hein2024_noc': 0.0, 'itzhak2016': 0.0}
W_EXT_ORG, W_EXT_NEU = 0.5, 0.8          # fonti esterne messe dall'utente in data/esterni/fonti (proteomica assonale iPSC, ecc.)
STRONG_ORG = {'deeploc', 'uniprot_forte', 'hpa_alta', 'mitocarta', 'goa_exp'}
STRONG_NEU = {'syngo', 'goa_exp', 'uniprot_forte'}     # + ogni fonte esterna; silver_oligo è ESCLUSA dalle metriche
known = set(PESI_ORG) | set(PESI_NEU)
ext_src = sorted(set(obs['fonte']) - known)
STRONG_NEU |= set(ext_src)
pesi = pd.DataFrame([(s, PESI_ORG.get(s, W_EXT_ORG), PESI_NEU.get(s, W_EXT_NEU), s in STRONG_ORG, s in STRONG_NEU) for s in sorted(set(obs['fonte']))],
                    columns=['fonte', 'peso_Asse_A', 'peso_Asse_B', 'forte_A', 'forte_B']).set_index('fonte')
print('Pesi effettivi per fonte:\n', pesi.to_string())

cidx = {c: i for i, c in enumerate(ALL)}
V_org = np.zeros((N, NO))                  # voti pesati dei positivi → quote
Y_neu = np.zeros((N, NN)); COV_neu = np.zeros((N, NN), bool)
P1, P0 = np.zeros((N, len(ALL)), bool), np.zeros((N, len(ALL)), bool)       # positivi/zeri delle fonti FORTI → REF
for s, g in obs.groupby('fonte'):
    ci, pi, o = g['classe'].map(cidx).to_numpy(), g['pid'].to_numpy(), g['obs'].to_numpy()
    a, b = ci < NO, ci >= NO
    wa, wb = PESI_ORG.get(s, W_EXT_ORG), PESI_NEU.get(s, W_EXT_NEU)
    if wa > 0:
        np.add.at(V_org, (pi[a & (o == 1)], ci[a & (o == 1)]), wa)
    if wb > 0:
        COV_neu[pi[b], ci[b] - NO] = True
        np.maximum.at(Y_neu, (pi[b], ci[b] - NO), wb * o[b])
    for strong, sel in [(s in STRONG_ORG, a), (s in STRONG_NEU, b)]:
        if strong:
            P1[pi[sel & (o == 1)], ci[sel & (o == 1)]] = True
            P0[pi[sel & (o == 0)], ci[sel & (o == 0)]] = True

M_org = (V_org.sum(1) > 0) & is_train                                          # righe con almeno un organello noto
Y_org = V_org / np.maximum(V_org.sum(1, keepdims=True), 1e-9)
neur, oligo = prot['is_neuronal'].to_numpy() == 1, prot['is_oligo'].to_numpy() == 1
elig = np.stack([neur, neur, neur, neur, oligo], 1)          # assone/dendrite/sinapsi: proteine neuronali · mielina: oligodendrocitarie
M_neu = COV_neu & (elig | (Y_neu > 0)) & is_train[:, None]
Y_neu = np.where(M_neu, Y_neu, 0.0)

REF = np.where(P1, 1.0, np.where(P0, 0.0, np.nan))
REF[~is_train] = np.nan
REF[:, NO:] = np.where(elig | (REF[:, NO:] == 1), REF[:, NO:], np.nan)       # zeri neuronali solo per proteine neuronali/oligodendrocitarie
Y_test = prot[['test_' + c for c in ORG]].to_numpy(dtype=float)               # test HPA indipendente (solo organelli visibili al microscopio)
Y_test[~is_test] = np.nan

n_pos = np.nansum(REF == 1, 0); n_neg = np.nansum(REF == 0, 0)
balance = pd.DataFrame({'asse': ['A'] * NO + ['B'] * NN, 'positivi_REF': n_pos.astype(int), 'negativi_REF': n_neg.astype(int),
                        'prevalenza': (n_pos / np.maximum(n_pos + n_neg, 1)).round(3), 'affidabile': n_pos >= MIN_EVAL_POS}, index=ALL)
RELIABLE = balance['affidabile'].to_numpy()
print('\nBilanciamento delle classi (riferimento = sole fonti forti):\n', balance.to_string())
print('\nSoft label: proteine con organello noto', int(M_org.sum()), '· elementi neuronali etichettati', int(M_neu.sum()))
print('Classi NON affidabili (< %d positivi): %s → addestrate, ma metriche marcate' % (MIN_EVAL_POS, [c for c, r in zip(ALL, RELIABLE) if not r] or 'nessuna'))

## Fold per omologia con stratificazione iterativa

In [ ]:
# ---------- Fold: gruppi di omologia + stratificazione iterativa multi-label (Sechidis et al., 2011) -----------------------------------
# Il piano propone MultilabelStratifiedKFold: bene per la rarità delle classi, ma ignora l'OMOLOGIA (paraloghi in fold diversi = leakage:
# CHRNA1 in train e CHRNA4 in test). Qui l'algoritmo di Sechidis lavora su GRUPPI di proteine omologhe, non su proteine singole.
# Con N positivi < k (es. mielina) NESSUNA stratificazione può mettere un positivo in ogni fold: le metriche si calcolano quindi sulle
# predizioni out-of-fold RIUNITE, mai per singolo fold (un AUC su un fold senza positivi non esiste).
def homology_groups(seqs, emb, family):
    if shutil.which('mmseqs'):                                           # migliore: cluster al 30 % di identità
        import subprocess, tempfile
        with tempfile.TemporaryDirectory() as td:
            fa = Path(td) / 'p.fa'
            fa.write_text(''.join(f'>{i}\n{s}\n' for i, s in enumerate(seqs)))
            subprocess.run(['mmseqs', 'easy-cluster', str(fa), f'{td}/c', f'{td}/t', '--min-seq-id', '0.3', '-c', '0.5', '--cov-mode', '1', '-v', '0'], check=True)
            cl = pd.read_csv(f'{td}/c_cluster.tsv', sep='\t', header=None, names=['rep', 'mem']).astype(int)
        return pd.factorize(cl.set_index('mem')['rep'].reindex(range(len(seqs))))[0]
    k = max(50, len(seqs) // 8)
    km = KMeans(k, n_init=3, random_state=SEED).fit_predict(normalize(PCA(64, random_state=SEED).fit_transform(emb.astype(np.float32))))
    fam = pd.Series(family).fillna('').str.split(',').str[-1].str.strip().str.lower().to_numpy()
    key = np.where(fam != '', 'f:' + fam, 'k:' + km.astype(str))
    sizes = pd.Series(key).map(pd.Series(key).value_counts()).to_numpy()
    key = np.where(sizes > max(30, 0.03 * len(seqs)), key + '|' + km.astype(str), key)      # famiglie enormi (GPCR, chinasi) divise per cluster
    return pd.factorize(key)[0]

def iterative_group_folds(Y, groups, k, seed=SEED):
    """Stratificazione iterativa su gruppi: assegna per primi i gruppi con l'etichetta più rara al fold che ne ha più bisogno."""
    rng = np.random.RandomState(seed)
    u, inv = np.unique(groups, return_inverse=True)
    G, L = len(u), Y.shape[1]
    YG = np.zeros((G, L)); np.add.at(YG, inv, Y)
    size = np.bincount(inv, minlength=G).astype(float)
    want_lab, want_size = np.tile(YG.sum(0) / k, (k, 1)), np.full(k, size.sum() / k)
    cnt, fsize, fold_of, free = np.zeros((k, L)), np.zeros(k), -np.ones(G, int), np.ones(G, bool)
    def put(g, f):
        fold_of[g] = f; free[g] = False; cnt[f] += YG[g]; fsize[f] += size[g]
    while free.any():
        left = YG[free].sum(0)
        cand = np.flatnonzero(left > 0)
        if len(cand) == 0:
            for g in rng.permutation(np.flatnonzero(free)):
                put(g, int(np.argmax(want_size - fsize)))
            break
        l = cand[np.argmin(left[cand])]
        for g in rng.permutation(np.flatnonzero(free & (YG[:, l] > 0))):
            need = want_lab[:, l] - cnt[:, l]
            fs = np.flatnonzero(need == need.max())
            if len(fs) > 1:
                sz = (want_size - fsize)[fs]
                fs = fs[sz == sz.max()]
            put(g, int(rng.choice(fs)))
    return fold_of[inv]

groups = homology_groups(seqs, E_seq, prot['family'].to_numpy())
Ystrat = np.zeros((N, len(ALL)))
Ystrat[:, :NO] = (Y_org >= 0.3) & M_org[:, None]
Ystrat[:, NO:] = (Y_neu >= 0.5) & M_neu
Ystrat = np.maximum(Ystrat, np.nan_to_num(REF) == 1)
fold = -np.ones(N, int)
fold[is_train] = iterative_group_folds(Ystrat[is_train], groups[is_train], N_FOLDS)
print(f'gruppi di omologia: {len(np.unique(groups[is_train])):,} · il gruppo più grande ha {np.bincount(groups[is_train]).max()} proteine')
fold_pos = pd.DataFrame({f'fold{k}': (np.nan_to_num(REF)[fold == k] == 1).sum(0) for k in range(N_FOLDS)}, index=ALL)
print('Positivi REF per fold e classe:\n', fold_pos.to_string())
print('Classi con un fold senza positivi (le metriche saranno sulle OOF riunite):', fold_pos.index[(fold_pos == 0).any(axis=1)].tolist() or 'nessuna')

## Metriche

In [ ]:
# ---------- metriche (AUC-ROC, AUPRC, MCC, ECE) ----------------------------------------------------------------------------------------
def best_thr(y, p):
    """Soglia che massimizza l'MCC (scelta SOLO sul validation, poi applicata al test)."""
    if len(np.unique(y)) < 2:
        return 0.5
    grid = np.unique(np.quantile(p, np.linspace(0.02, 0.98, 49)))
    return float(grid[np.argmax([matthews_corrcoef(y, p >= t) for t in grid])])

def class_metrics(p, y, thr=None, b=None):
    ok = ~np.isnan(y)
    y, p = y[ok], p[ok]
    b = None if b is None else b[ok]
    npos, nneg = int((y == 1).sum()), int((y == 0).sum())
    r = dict(n_pos=npos, n_neg=nneg, auroc=np.nan, auprc=np.nan, mcc=np.nan, reliable=npos >= MIN_EVAL_POS and nneg >= MIN_EVAL_POS)
    if npos and nneg:
        r['auroc'], r['auprc'] = roc_auc_score(y, p), average_precision_score(y, p)
        if b is not None:
            r['mcc'] = matthews_corrcoef(y, b)
        elif thr is not None:
            r['mcc'] = matthews_corrcoef(y, p >= thr)
    return r

def table(P, Y, names, rows=None, thr=None, Bin=None):
    """P, Y: (N, C). Una riga per classe + media macro sulle sole classi affidabili."""
    rows = np.ones(len(P), bool) if rows is None else rows
    t = pd.DataFrame({n: class_metrics(P[rows, i], Y[rows, i], None if thr is None else thr[i], None if Bin is None else Bin[rows, i]) for i, n in enumerate(names)}).T
    for c in ['auroc', 'auprc', 'mcc']:
        t[c] = t[c].astype(float)
    return t

def macro(t, col='auroc'):
    return float(t.loc[t['reliable'].astype(bool), col].mean())

def ece_binary(p, y, bins=10):
    ok = ~np.isnan(y); p, y = p[ok], y[ok]
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & ((p < hi) | (hi == 1))
        if m.any():
            e += abs(y[m].mean() - p[m].mean()) * m.mean()
    return e

def ece_continuous(pred, meas, bins=10):
    """ECE per un bersaglio continuo (frazione misurata): |media misurata − media predetta| per bin di predizione."""
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pred >= lo) & ((pred < hi) | (hi == 1))
        if m.any():
            e += abs(meas[m].mean() - pred[m].mean()) * m.mean()
    return e

def fold_rows(k):
    return np.flatnonzero((fold >= 0) & ~np.isin(fold, [k, (k + 1) % N_FOLDS])), np.flatnonzero(fold == (k + 1) % N_FOLDS), np.flatnonzero(fold == k)
EXT_ROWS = np.flatnonzero(is_test)           # proteine del test HPA: predette da tutti i fold, mai usate per addestrare

## Riferimenti: prevalenza, regressione logistica, DeepLoc-style

In [ ]:
# ---------- Tre riferimenti: prevalenza · regressione logistica · "stile DeepLoc 2.0" ---------------------------------------------------
# Tutti addestrati sul riferimento binario REF (sole fonti forti) con gli STESSI fold del modello finale.
#  · prevalenza   : predice la frequenza della classe (AUC = 0,5 per costruzione) — il pavimento
#  · regressione logistica: vede TUTTE le informazioni (ESM, frammenti, card, struttura), compresse con PCA — il lineare forte
#  · DeepLoc-style: solo ESM-2 medio → MLP con BCE, la ricetta di DeepLoc 2.0 senza testo né struttura
# Il modello finale in più usa le soft label di tutte le fonti pesate, la card, i frammenti separati e la struttura.
from sklearn.preprocessing import StandardScaler

def build_lr_features():
    tr_all = is_train
    parts = []
    for X, k in [(E_seq, 256), (E_frag, 128), (E_text['protein'], 64)]:
        X = X.astype(np.float32)
        parts.append(PCA(k, random_state=SEED).fit(X[tr_all]).transform(X))
    Z = np.hstack(parts + [F_struct, af_missing])
    return StandardScaler().fit(Z[tr_all]).transform(Z).astype(np.float32)

Z_lr = build_lr_features()
C_all = len(ALL)
P_prev, P_lr, P_dl = (np.full((N, C_all), np.nan) for _ in range(3))
ext_acc = {n: np.zeros((N, C_all)) for n in ['prev', 'lr', 'dl']}

def fit_mlp_baseline(tr, va, seed):
    torch.manual_seed(seed)
    x = torch.tensor((E_seq.astype(np.float32) - E_seq[is_train].astype(np.float32).mean(0)) / (E_seq[is_train].astype(np.float32).std(0) + 1e-4), device=DEVICE)
    y = torch.tensor(np.nan_to_num(REF), dtype=torch.float32, device=DEVICE); m = torch.tensor(~np.isnan(REF), dtype=torch.float32, device=DEVICE)
    net = nn.Sequential(nn.Dropout(0.3), nn.Linear(ESM_DIM, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, C_all)).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-2)
    posw = torch.tensor(np.clip((np.nansum(REF[tr] == 0, 0) + 1) / (np.nansum(REF[tr] == 1, 0) + 1), 1, 20), dtype=torch.float32, device=DEVICE)
    def loss_of(idx):
        l = F.binary_cross_entropy_with_logits(net(x[idx]), y[idx], pos_weight=posw, reduction='none') * m[idx]
        return l.sum() / m[idx].sum().clamp(min=1)
    best, state, bad = 1e9, None, 0
    for ep in range(60):
        net.train()
        perm = np.random.permutation(tr)
        for b in range(0, len(perm), 128):
            opt.zero_grad(); loss_of(torch.as_tensor(perm[b:b + 128], device=DEVICE)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            v = loss_of(torch.as_tensor(va, device=DEVICE)).item()
        if v < best - 1e-4:
            best, state, bad = v, copy.deepcopy(net.state_dict()), 0
        else:
            bad += 1
            if bad >= 8:
                break
    net.load_state_dict(state); net.eval()
    return lambda idx: torch.sigmoid(net(x[torch.as_tensor(idx, device=DEVICE)])).detach().cpu().numpy()

for k in range(N_FOLDS):
    tr, va, te = fold_rows(k)
    trva = np.concatenate([tr, va])                       # la regressione logistica non ha bisogno di validation: usa un fold in più (baseline più forte)
    pred_dl = fit_mlp_baseline(tr, va, SEED + k)
    for c in range(C_all):
        yy = REF[trva, c]; ok = ~np.isnan(yy)
        pr = float(np.nan_to_num(np.nanmean(REF[tr, c]), nan=0.0)) if ok.any() else 0.0
        P_prev[te, c] = pr; ext_acc['prev'][EXT_ROWS, c] += pr / N_FOLDS
        if (yy[ok] == 1).sum() >= 2 and (yy[ok] == 0).sum() >= 2:
            lr = LogisticRegression(C=0.05, class_weight='balanced', max_iter=400).fit(Z_lr[trva][ok], yy[ok])
            P_lr[te, c] = lr.predict_proba(Z_lr[te])[:, 1]; ext_acc['lr'][EXT_ROWS, c] += lr.predict_proba(Z_lr[EXT_ROWS])[:, 1] / N_FOLDS
        else:
            P_lr[te, c] = pr; ext_acc['lr'][EXT_ROWS, c] += pr / N_FOLDS
    P_dl[te] = pred_dl(te); ext_acc['dl'][EXT_ROWS] += pred_dl(EXT_ROWS) / N_FOLDS
    print(f'baseline fold {k + 1}/{N_FOLDS} fatto')
for n, P in [('prev', P_prev), ('lr', P_lr), ('dl', P_dl)]:
    P[EXT_ROWS] = ext_acc[n][EXT_ROWS]
t_lr, t_dl = table(P_lr, REF, ALL), table(P_dl, REF, ALL)
print(f'OOF macro AUROC (classi affidabili)  logistica {macro(t_lr):.3f} · DeepLoc-style {macro(t_dl):.3f} · prevalenza 0.500')

## Architettura e loss

In [ ]:
# =====================================================================================================================================
#  NeuroLoc-Δ v2 · architettura "Double-Head" (piano, Sub-Task 2) con le correzioni emerse dall'analisi critica
#
#   E_seq  (1280) ─ proj_seq ──┐                                    ┌─ head_org (256→9)  Softmax  ← Asse A, KL + λ·focale su soft label
#   E_frag (3×1280) ─ proj_frag┼─ 4 token K/V ─┐                    │
#   E_text (1024) ─[gate]─[card-dropout]─ proj_text ─ Q ─ CrossAttn ─ fusion(→512→256) = repr_256
#   F_struct+af_missing ─ proj_struct (64) ───────────────────────┘ └─ head_neu (256→5)  Sigmoid ← Asse B, ASL + pesi class-balanced
#   Loss totale = Kendall(L_org, L_ASL) + β·SupCon(repr_256, classi rare)
#
#  CORREZIONI al progetto di riferimento del piano (spiegate nelle celle di analisi):
#   1. La "cross-attention" del piano ha UNA query e UNA chiave: il softmax su un solo elemento vale 1 → l'output non dipende dalla query,
#      è una semplice mappa lineare. Qui K/V sono QUATTRO token (sequenza intera + N + centro + C): la card sceglie quale pezzo interrogare.
#   2. gate_w nel piano è usato solo in inferenza → mai addestrato (resta 1.0). Qui il gate lavora anche in training (e ha un bias).
#   3. "KL" nel piano è in realtà cross-entropy: il valore minimo è l'entropia delle soft label, non 0. Con i pesi di Kendall (≈ 1/L) l'offset conta:
#      qui è una KL vera (≥ 0).
#   4. SupCon del piano media su TUTTI gli anchor, anche quelli senza coppia positiva → diluisce la loss; qui solo gli anchor validi.
# =====================================================================================================================================
class KLFocalLoss(nn.Module):
    """Asse A: KL(y ‖ p) + λ · Σ (1−p)^γ (−log p).  Il secondo termine è una penalità di confidenza (come il label smoothing), non una focal loss classica."""
    def __init__(self, gamma=2.0, lam=0.1, eps=1e-8):
        super().__init__(); self.gamma, self.lam, self.eps = gamma, lam, eps
    def forward(self, logits, y, rowmask):
        lp = F.log_softmax(logits, 1)
        kl = (y * (torch.log(y.clamp_min(self.eps)) - lp)).sum(1)
        foc = -((1 - lp.exp()).pow(self.gamma) * lp).sum(1)
        return ((kl + self.lam * foc) * rowmask).sum() / rowmask.sum().clamp_min(1)

class AsymmetricLoss(nn.Module):
    """Asse B: Asymmetric Loss (Ben-Baruch et al., 2020) con maschera di eleggibilità e pesi class-balanced per classe sui positivi."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, pos_weights=None):
        super().__init__(); self.gn, self.gp, self.clip, self.eps, self.pw = gamma_neg, gamma_pos, clip, eps, pos_weights
    def forward(self, x, y, mask):
        p = torch.sigmoid(x)
        pn = (1 - p + self.clip).clamp(max=1.0) if self.clip else 1 - p     # (nel piano 03 questo ramo usa un walrus che genera NameError: qui no)
        lpos = y * torch.log(p.clamp_min(self.eps)) * (1 - p).pow(self.gp)
        if self.pw is not None:
            lpos = lpos * self.pw
        lneg = (1 - y) * torch.log(pn.clamp_min(self.eps)) * p.pow(self.gn)
        rows = (mask.sum(1) > 0).float().sum().clamp_min(1)
        return -((lpos + lneg) * mask).sum() / rows

class MultiTaskUncertaintyLoss(nn.Module):
    """Kendall et al. 2018: L = Σ_i exp(−s_i)·L_i + s_i.  Con questa forma il peso di equilibrio è 1/L_i: lo si limita a s ∈ [−3, 3] per evitare
    che un task quasi a zero (es. ASL con poche righe) monopolizzi il gradiente."""
    def __init__(self):
        super().__init__(); self.log_vars = nn.Parameter(torch.zeros(2))
    def forward(self, l_org, l_neu):
        s = self.log_vars.clamp(-3, 3)
        return torch.exp(-s[0]) * l_org + s[0] + torch.exp(-s[1]) * l_neu + s[1]

class SupConLoss(nn.Module):
    """Supervised Contrastive (Khosla et al., 2020) sulle sole classi rare dell'Asse B. labels: (B, R) binari."""
    def __init__(self, temperature=0.07):
        super().__init__(); self.t = temperature
    def forward(self, feats, labels):
        B = feats.shape[0]
        z = F.normalize(feats, dim=1)
        sim = z @ z.T / self.t
        sim = sim - sim.max(1, keepdim=True).values.detach()
        eye = torch.eye(B, device=feats.device)
        pos = ((labels @ labels.T) > 0).float() * (1 - eye)
        logp = sim - torch.log((torch.exp(sim) * (1 - eye)).sum(1, keepdim=True) + 1e-8)
        npos = pos.sum(1)
        valid = npos > 0
        if not valid.any():
            return feats.sum() * 0.0
        return -((pos * logp).sum(1) / npos.clamp_min(1))[valid].mean()

class DynamicTextGate(nn.Module):
    """g = σ(w·(6·x − 3) + b) con x = log(1+parole)/log(1+500). Poche parole di testo UniProt vero → il testo pesa poco. Scala log: i conteggi hanno coda lunga."""
    def __init__(self):
        super().__init__(); self.w = nn.Parameter(torch.tensor(1.0)); self.b = nn.Parameter(torch.tensor(0.0))
    def forward(self, n_words):
        x = torch.log1p(n_words) / math.log1p(500.0)
        return torch.sigmoid(self.w * (6 * x - 3) + self.b)

def block(i, o, p):
    return nn.Sequential(nn.Linear(i, o), nn.LayerNorm(o), nn.GELU(), nn.Dropout(p))

class MMCANSubCellRobust(nn.Module):
    def __init__(self, esm_dim, frag_dim, bert_dim=1024, struct_dim=23, n_org=9, n_neu=5, proj_dim=512, n_heads=8, dropout=0.3, card_drop_prob=0.35):
        super().__init__()
        assert frag_dim % 3 == 0
        self.card_drop_prob = card_drop_prob
        self.null_card_emb = nn.Parameter(torch.zeros(1, bert_dim)); nn.init.normal_(self.null_card_emb, std=0.02)
        self.gate = DynamicTextGate()
        self.proj_seq = block(esm_dim, proj_dim, dropout * 0.5)
        self.proj_frag = block(frag_dim // 3, proj_dim, dropout * 0.5)          # condivisa dai 3 frammenti; il "tipo" li distingue
        self.proj_text = block(bert_dim, proj_dim, dropout * 0.5)
        self.proj_struct = block(struct_dim + 1, 64, dropout * 0.3)             # +1 = flag af_missing (Bug 5)
        self.kv_type = nn.Parameter(torch.randn(1, 4, proj_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(proj_dim, n_heads, dropout=dropout * 0.5, batch_first=True)
        self.attn_norm = nn.LayerNorm(proj_dim)
        self.fusion = nn.Sequential(nn.Linear(proj_dim * 3 + 64, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
                                    nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout * 0.5))
        self.head_org, self.head_neu = nn.Linear(256, n_org), nn.Linear(256, n_neu)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
        self.last_attn = None

    def forward(self, E_seq, E_frag, E_text, F_struct, card_n_words=None, af_missing_mask=None, use_gate=True):
        B = E_seq.shape[0]
        if af_missing_mask is None:
            af_missing_mask = F_struct.new_zeros(B, 1)
        null = self.null_card_emb.expand(B, -1)
        if use_gate and card_n_words is not None:
            g = self.gate(card_n_words).unsqueeze(1)
            E_text = g * E_text + (1 - g) * null
        if self.training and self.card_drop_prob > 0:
            keep = (torch.rand(B, 1, device=E_seq.device) > self.card_drop_prob).float()
            E_text = keep * E_text + (1 - keep) * null                          # Card-Dropout con vettore nullo apprendibile
        kv = torch.cat([self.proj_seq(E_seq).unsqueeze(1), self.proj_frag(E_frag.reshape(B, 3, -1))], 1) + self.kv_type      # (B,4,D)
        H_text = self.proj_text(E_text)
        H_struct = self.proj_struct(torch.cat([F_struct, af_missing_mask], 1))
        att, w = self.cross_attn(H_text.unsqueeze(1), kv, kv, need_weights=True)
        self.last_attn = w.detach().squeeze(1)                                  # (B,4): quale pezzo (intera, N, centro, C) la card interroga
        attended = self.attn_norm(att.squeeze(1) + H_text)
        r = self.fusion(torch.cat([attended, H_text, H_struct, kv.mean(1)], 1))
        lo, ln = self.head_org(r), self.head_neu(r)
        return lo, F.softmax(lo, 1), ln, torch.sigmoid(ln), r

class MMCANStudent(nn.Module):
    """Sequenza + AlphaFold, nessun testo: il modello di produzione per proteine orfane (stessa Double-Head, stesso af_missing)."""
    def __init__(self, esm_dim, frag_dim, struct_dim=23, n_org=9, n_neu=5, proj_dim=512, dropout=0.3):
        super().__init__()
        self.proj_seq, self.proj_frag = block(esm_dim, proj_dim, dropout * 0.5), block(frag_dim, proj_dim, dropout * 0.5)
        self.proj_struct = block(struct_dim + 1, 64, dropout * 0.3)
        self.fusion = nn.Sequential(nn.Linear(proj_dim * 2 + 64, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
                                    nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU())
        self.head_org, self.head_neu = nn.Linear(256, n_org), nn.Linear(256, n_neu)
    def forward(self, E_seq, E_frag, F_struct, af_missing_mask=None):
        if af_missing_mask is None:
            af_missing_mask = F_struct.new_zeros(E_seq.shape[0], 1)
        r = self.fusion(torch.cat([self.proj_seq(E_seq), self.proj_frag(E_frag), self.proj_struct(torch.cat([F_struct, af_missing_mask], 1))], 1))
        lo, ln = self.head_org(r), self.head_neu(r)
        return lo, F.softmax(lo, 1), ln, torch.sigmoid(ln), r

def kd_loss(zo_T, zn_T, r_T, zo_S, zn_S, r_S, T=4.0, beta=0.01, neu_mask=None):
    """Dual-Head Feature & Logit Distillation: T²·KL(softmax) + BCE(sigmoid) + β·MSE(repr). Softmax e sigmoid hanno due formule diverse."""
    kl = F.kl_div(F.log_softmax(zo_S / T, 1), F.softmax(zo_T / T, 1), reduction='none').sum(1).mean() * T * T
    b = F.binary_cross_entropy_with_logits(zn_S / T, torch.sigmoid(zn_T / T), reduction='none')
    b = (b * neu_mask).sum() / neu_mask.sum().clamp_min(1) if neu_mask is not None else b.mean()
    return kl + b + beta * F.mse_loss(r_S, r_T)

## Addestramento, predizione, Temperature Scaling

In [ ]:
# ---------- dati sul dispositivo, campionatore, funzione di addestramento, predizione, temperature scaling, CV -----------------------------
def T_(a, dt=torch.float32):
    return torch.as_tensor(np.asarray(a), dtype=dt, device=DEVICE)

TS = dict(seq=T_(E_seq), frag=T_(E_frag), struct=T_(F_struct), afm=T_(af_missing), nw=T_(card_n_annot),
          yorg=T_(Y_org), morg=T_(M_org.astype(np.float32)), yneu=T_(Y_neu), mneu=T_(M_neu.astype(np.float32)))
TXT = {m: T_(E_text[m]) for m in CARD_MODES}
OUT_ROWS = np.flatnonzero(fold == -1)                    # test HPA + proteine senza etichetta: predette da tutti i fold e mediate
UNL_ROWS = np.flatnonzero(is_train & (fold == -1))       # senza etichetta (per la distillazione dello Student)
print('struttura: STRUCT_DIM =', STRUCT_DIM, '· righe da predire fuori fold:', len(OUT_ROWS))

def fwd(net, idx, kind, mode='protein', use_gate=True, null_text=False):
    s, f, st, am = TS['seq'][idx], TS['frag'][idx], TS['struct'][idx], TS['afm'][idx]
    if kind == 'student':
        return net(s, f, st, am)
    tx = net.null_card_emb.expand(len(idx), -1) if null_text else TXT[mode][idx]
    return net(s, f, tx, st, TS['nw'][idx], am, use_gate=use_gate)

def make_net(kind, card_drop):
    h = HYPER
    if kind == 'student':
        return MMCANStudent(ESM_DIM, 3 * ESM_DIM, STRUCT_DIM, NO, NN, h['proj_dim'], h['dropout']).to(DEVICE)
    return MMCANSubCellRobust(ESM_DIM, 3 * ESM_DIM, BERT_DIM, STRUCT_DIM, NO, NN, h['proj_dim'], h['n_heads'], h['dropout'], card_drop).to(DEVICE)

def fit(kind, tr, va, mode='protein', card_drop=None, beta_supcon=None, teacher=None, seed=0, verbose=False):
    h = HYPER
    card_drop = h['card_drop_prob'] if card_drop is None else card_drop
    beta_supcon = h['beta_supcon'] if beta_supcon is None else beta_supcon
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    npos = ((Y_neu[tr] >= 0.5) & M_neu[tr]).sum(0)                              # N_j sul training
    cb = (1 - h['cb_beta']) / (1 - h['cb_beta'] ** np.maximum(npos, 1))          # W_j = (1−β)/(1−β^N_j)  (Cui et al. 2019)
    cb = np.clip(cb / np.median(cb), 0.5, 10.0)                                  # ri-scalato: il W_j grezzo del piano rimpicciolirebbe TUTTI i positivi ~100× rispetto ai negativi
    asl = AsymmetricLoss(h['asl_gneg'], h['asl_gpos'], h['asl_clip'], pos_weights=T_(cb))
    klf, unc, sup = KLFocalLoss(h['kl_gamma'], h['kl_lambda']), MultiTaskUncertaintyLoss().to(DEVICE), SupConLoss(h['supcon_temp'])
    rare = np.flatnonzero(npos < h['supcon_rare_max'])
    net = make_net(kind, card_drop)
    opt = torch.optim.AdamW([{'params': net.parameters(), 'lr': h['lr'], 'weight_decay': h['wd']},
                             {'params': unc.parameters(), 'lr': h['lr_logvar'], 'weight_decay': 0.0}])
    rows = np.concatenate([tr, UNL_ROWS]) if (kind == 'student' and KD_USE_UNLABELED) else tr
    spe = math.ceil(len(rows) / h['batch'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[h['lr'], h['lr_logvar']], total_steps=spe * h['epochs'], pct_start=0.1)
    # campionamento: le proteine con un positivo raro compaiono più spesso → ogni batch ha coppie positive per SupCon
    boost = ((Y_neu[rows][:, rare] >= 0.5) & M_neu[rows][:, rare]) * (50.0 / np.maximum(npos[rare], 1)) ** 0.5 if len(rare) else np.zeros((len(rows), 1))
    p = 1.0 + boost.sum(1); p = p / p.sum()
    best, state, bad = 1e9, None, 0
    for ep in range(h['epochs']):
        net.train(); tot = 0.0
        for b in range(spe):
            idx = torch.as_tensor(rng.choice(rows, size=min(h['batch'], len(rows)), p=p), device=DEVICE)
            out = fwd(net, idx, kind, mode)
            lo, _, ln, _, r = out
            l_org = klf(lo, TS['yorg'][idx], TS['morg'][idx]); l_neu = asl(ln, TS['yneu'][idx], TS['mneu'][idx])
            if kind == 'teacher':
                loss = unc(l_org, l_neu)
                if beta_supcon > 0 and len(rare):
                    lab = ((TS['yneu'][idx][:, rare] >= 0.5).float() * TS['mneu'][idx][:, rare])
                    loss = loss + beta_supcon * sup(r, lab)
            else:                                                                  # Student: distillazione + supervisione dura
                with torch.no_grad():
                    tout = fwd(teacher, idx, 'teacher', mode)
                loss = kd_loss(tout[0], tout[2], tout[4], lo, ln, r, h['T_kd'], h['beta_feat'], TS['mneu'][idx]) + h['lam_hard'] * (l_org + l_neu)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item()
        net.eval()
        with torch.no_grad():                                                      # validation: loss dura non pesata (confrontabile tra epoche)
            v, n = 0.0, 0
            for s in range(0, len(va), 2048):
                idx = torch.as_tensor(va[s:s + 2048], device=DEVICE)
                o = fwd(net, idx, kind, mode)
                v += (klf(o[0], TS['yorg'][idx], TS['morg'][idx]) + asl(o[2], TS['yneu'][idx], TS['mneu'][idx])).item() * len(idx); n += len(idx)
            v /= n
        if v < best - 1e-4:
            best, state, bad = v, copy.deepcopy(net.state_dict()), 0
        else:
            bad += 1
        if verbose and ep % 5 == 0:
            ls = unc.log_vars.detach().cpu().numpy()
            print(f'  ep {ep:02d} train {tot / spe:.3f} val {v:.3f} · σ²_org={math.exp(ls[0]):.2f} σ²_neu={math.exp(ls[1]):.2f}')
        if bad >= h['patience']:
            break
    net.load_state_dict(state); net.eval()
    return net, unc

@torch.no_grad()
def predict(net, rows, kind, mode='protein', use_gate=True, null_text=False, chunk=2048):
    net.eval(); LO, LN, R, A = [], [], [], []
    for s in range(0, len(rows), chunk):
        idx = torch.as_tensor(rows[s:s + chunk], device=DEVICE)
        o = fwd(net, idx, kind, mode, use_gate, null_text)
        LO.append(o[0].cpu().numpy()); LN.append(o[2].cpu().numpy()); R.append(o[4].cpu().numpy())
        if kind == 'teacher':
            A.append(net.last_attn.cpu().numpy())
    return np.concatenate(LO), np.concatenate(LN), np.concatenate(R), (np.concatenate(A) if A else None)

def fit_temperature(lo, ln, rows):
    """Temperature Scaling (Guo et al. 2017) sul validation: T_org su cross-entropy soft (softmax), T_neu su BCE mascherata (sigmoidi). LBFGS."""
    out = []
    for kind in ['org', 'neu']:
        t = torch.zeros(1, requires_grad=True)
        z = torch.tensor(lo if kind == 'org' else ln, dtype=torch.float32)
        y = torch.tensor((Y_org if kind == 'org' else Y_neu)[rows], dtype=torch.float32)
        m = torch.tensor((M_org[:, None] if kind == 'org' else M_neu)[rows], dtype=torch.float32)
        opt = torch.optim.LBFGS([t], lr=0.1, max_iter=30)
        def closure():
            opt.zero_grad()
            if kind == 'org':
                l = -(y * F.log_softmax(z / t.exp(), 1)).sum(1) * m[:, 0]
                l = l.sum() / m.sum().clamp_min(1)
            else:
                l = (F.binary_cross_entropy_with_logits(z / t.exp(), y, reduction='none') * m).sum() / m.sum().clamp_min(1)
            l.backward(); return l
        if m.sum() > 20:
            opt.step(closure)
        out.append(float(t.exp().clamp(0.3, 5.0)))
    return out

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
sigmoid_np = lambda z: 1 / (1 + np.exp(-z))

## Driver di cross-validation

In [ ]:
# ---------- driver di cross-validation: addestra, calibra, predice (varianti full / card-free / senza gate) --------------------------------
def run_cv(name, kind='teacher', mode='protein', card_drop=None, beta_supcon=None, teacher_states=None, teacher_mode='protein', seeds=(0,), keep_states=False):
    t0 = time.time()
    variants = ['full', 'free', 'nogate'] if kind == 'teacher' else ['full']
    mk = lambda *s: np.zeros(s)
    R = {v: dict(po_raw=mk(N, NO), po=mk(N, NO), pn_raw=mk(N, NN), pn=mk(N, NN)) for v in variants}
    ext_n = {v: 0 for v in variants}
    BIN = np.zeros((N, len(ALL)), np.int8)                         # predizioni binarie con la soglia scelta sul validation del PROPRIO fold
    thr_all, Ts, states, REPR, ATT = [], [], [], np.zeros((N, 256), np.float32), np.zeros((N, 4), np.float32)
    for k in range(N_FOLDS):
        tr, va, te = fold_rows(k)
        eval_rows = {'va': va, 'te': te, 'out': OUT_ROWS}
        nets = []
        for sd in seeds:
            teacher = None
            if kind == 'student':
                teacher = make_net('teacher', 0.0); teacher.load_state_dict(teacher_states[k]); teacher.eval()
            net, _ = fit(kind, tr, va, mode=mode if kind == 'teacher' else teacher_mode, card_drop=card_drop, beta_supcon=beta_supcon,
                         teacher=teacher, seed=sd * 100 + k, verbose=(k == 0 and sd == seeds[0]))
            nets.append(net)
        if keep_states:
            states.append(copy.deepcopy(nets[0].state_dict()))
        pk = lambda rows, **kw: [np.mean(z, 0) if z[0] is not None else None for z in zip(*[predict(n, rows, kind, mode if kind == 'teacher' else teacher_mode, **kw) for n in nets])]
        lo_v, ln_v, _, _ = pk(va)
        T_o, T_n = fit_temperature(lo_v, ln_v, va)                 # temperatura dal validation di QUESTO fold
        Ts.append((T_o, T_n))
        pv = np.concatenate([softmax_np(lo_v / T_o), sigmoid_np(ln_v / T_n)], 1)
        thr_k = np.array([best_thr(REF[va, c][~np.isnan(REF[va, c])], pv[~np.isnan(REF[va, c]), c]) if (~np.isnan(REF[va, c])).sum() > 5 else 0.5 for c in range(len(ALL))])
        thr_all.append(thr_k)
        for v in variants:
            kw = dict(null_text=(v == 'free'), use_gate=(v != 'nogate')) if kind == 'teacher' else {}
            lo, ln, rep, att = pk(te, **kw)
            R[v]['po_raw'][te], R[v]['pn_raw'][te] = softmax_np(lo), sigmoid_np(ln)
            R[v]['po'][te], R[v]['pn'][te] = softmax_np(lo / T_o), sigmoid_np(ln / T_n)
            if v == 'full':
                REPR[te] = rep
                if att is not None:
                    ATT[te] = att
                BIN[te] = (np.concatenate([R[v]['po'][te], R[v]['pn'][te]], 1) >= thr_k).astype(np.int8)
            lo, ln, rep, att = pk(OUT_ROWS, **kw)                  # fuori fold: media dei fold
            for key, val in [('po_raw', softmax_np(lo)), ('po', softmax_np(lo / T_o)), ('pn_raw', sigmoid_np(ln)), ('pn', sigmoid_np(ln / T_n))]:
                R[v][key][OUT_ROWS] += val / N_FOLDS
            if v == 'full' and att is not None:
                ATT[OUT_ROWS] += att / N_FOLDS
        print(f'[{name}] fold {k + 1}/{N_FOLDS} · T_org={T_o:.2f} T_neu={T_n:.2f} · {time.time() - t0:.0f}s')
        del nets; gc.collect(); torch.cuda.empty_cache()
    return dict(name=name, kind=kind, R=R, BIN=BIN, thr=np.mean(thr_all, 0), T=np.array(Ts), repr=REPR, att=ATT, states=states)

def evaluate(res, variant='full', rows=None, calibrated=True):
    """Tabelle per Asse A e B sulle OOF riunite (riferimento REF)."""
    P = np.concatenate([res['R'][variant]['po' if calibrated else 'po_raw'], res['R'][variant]['pn' if calibrated else 'pn_raw']], 1)
    full = variant == 'full'                       # 'full': MCC con la soglia di ogni fold · altre varianti: soglia media dei fold
    tA = table(P[:, :NO], REF[:, :NO], ORG, rows, thr=None if full else res['thr'][:NO], Bin=res['BIN'][:, :NO] if full else None)
    tB = table(P[:, NO:], REF[:, NO:], NEU, rows, thr=None if full else res['thr'][NO:], Bin=res['BIN'][:, NO:] if full else None)
    return tA, tB

def evaluate_hpa(res, variant='full', rows=None):
    """Test indipendente HPA (solo organelli visibili al microscopio), probabilità medie dei 5 fold."""
    rr = EXT_ROWS if rows is None else rows
    return table(res['R'][variant]['po'], Y_test, ORG, np.isin(np.arange(N), rr), thr=res['thr'][:NO])

def summary(res, variant='full', rows=None):
    tA, tB = evaluate(res, variant, rows)
    return dict(AUC_A=macro(tA, 'auroc'), AUPRC_A=macro(tA, 'auprc'), MCC_A=macro(tA, 'mcc'), AUC_B=macro(tB, 'auroc'), AUPRC_B=macro(tB, 'auprc'), MCC_B=macro(tB, 'mcc'))

## Teacher principale e ablazioni (card full / protein / degraded · con/senza Card-Dropout · con/senza SupCon)

In [ ]:
# ---------- Teacher principale + ablazioni (Sub-Task 2, 3, 6a) ---------------------------------------------------------------------------
RUNS = {}
RUNS['teacher'] = run_cv('teacher', mode='protein', seeds=(0, 1), keep_states=True)          # card ripulita + Card-Dropout 0,35 + SupCon 0,1
if RUN_ABLATIONS:
    RUNS['card_full'] = run_cv('card_full', mode='full')                                      # limite SUPERIORE: card con localizzazione dentro
    RUNS['card_degraded'] = run_cv('card_degraded', mode='degraded')                          # limite INFERIORE: anche famiglie e classi funzionali → [FAM]
    RUNS['no_cardDrop'] = run_cv('no_cardDrop', card_drop=0.0)                                # senza Card-Dropout (serve a ST4)
    RUNS['no_supcon'] = run_cv('no_supcon', beta_supcon=0.0)                                  # senza SupCon (serve a ST6a)
rows_summary = {k: summary(v) for k, v in RUNS.items()}
abl = pd.DataFrame(rows_summary).T.round(3)
print('\nTabella di ablazione (OOF riunite, riferimento REF; AUC/AUPRC/MCC macro sulle classi affidabili):')
print(abl.to_string())
if RUN_ABLATIONS:
    d = lambda a, b: abl.loc[a, 'AUC_A'] - abl.loc[b, 'AUC_A']
    print(f"\nContributo del testo (Asse A, AUC): card_full − card_protein = {d('card_full', 'teacher'):+.3f} (quota di leakage dichiarato) · "
          f"card_protein − card_degraded = {d('teacher', 'card_degraded'):+.3f} (proxy indiretti: famiglie e funzioni)")
tA, tB = evaluate(RUNS['teacher'])
print('\nAsse A per classe:\n', tA.round(3).to_string()); print('\nAsse B per classe:\n', tB.round(3).to_string())
sh = evaluate_hpa(RUNS['teacher'])
print('\nTest HPA indipendente (organelli):\n', sh.round(3).to_string(), f"\nmacro AUROC {macro(sh):.3f} · AUPRC {macro(sh, 'auprc'):.3f}")

## Sub-Task 4 · Full / Card-Free / Low-Annotation e Dynamic Gating

In [ ]:
# ---------- Sub-Task 4: valutazione Full / Card-Free / Low-Annotation e Dynamic Gating ----------------------------------------------------
# Card-Free: E_text = null_card_emb (il vettore apprendibile, non zeri) su TUTTE le proteine.
# Low-Annotation: solo le proteine con < 15 parole di testo UniProt VERO (colonna card_n_annot, non card_n_words: vedi notebook 01).
low = (card_n_annot < LOW_ANNOT_WORDS)
print(f'proteine poco annotate: {int((low & is_train).sum())} nei fold · {int((low & is_test).sum())} nel test HPA')

def scenarios(res):
    out = {}
    for scen, var, rows in [('Full', 'full', None), ('Card-Free', 'free', None), ('Low-Annotation', 'full', low), ('Low-Annotation + Card-Free', 'free', low)]:
        for ax, t in zip(['A', 'B'], evaluate(res, var, rows)):
            out[(ax, scen)] = dict(AUROC=macro(t, 'auroc'), AUPRC=macro(t, 'auprc'), MCC=macro(t, 'mcc'), classi_valutabili=int(t['reliable'].astype(bool).sum()))
    return pd.DataFrame(out).T.round(3)

scen_with = scenarios(RUNS['teacher'])
print('\n=== Teacher CON Card-Dropout ===\n', scen_with.to_string())
if 'no_cardDrop' in RUNS:
    scen_wo = scenarios(RUNS['no_cardDrop'])
    print('\n=== Teacher SENZA Card-Dropout ===\n', scen_wo.to_string())
    for ax in ['A', 'B']:
        drop = lambda s: (s.loc[(ax, 'Full'), 'AUROC'] - s.loc[(ax, 'Card-Free'), 'AUROC']) / s.loc[(ax, 'Full'), 'AUROC']
        print(f"Asse {ax}: calo relativo di AUROC Full→Card-Free  con dropout {drop(scen_with):.1%} · senza dropout {drop(scen_wo):.1%}  "
              f"→ criterio del piano (< 10 % e migliore del twin senza dropout): {'RISPETTATO' if drop(scen_with) < 0.10 and drop(scen_with) < drop(scen_wo) else 'NON rispettato'}")

# Dynamic gate: stesso modello, gate acceso vs spento, sul sottoinsieme poco annotato
for ax, (tg, tn) in zip(['A', 'B'], zip(evaluate(RUNS['teacher'], 'full', low), evaluate(RUNS['teacher'], 'nogate', low))):
    print(f"Asse {ax} · poco annotate · gate ON  AUROC {macro(tg):.3f} AUPRC {macro(tg, 'auprc'):.3f} | gate OFF AUROC {macro(tn):.3f} AUPRC {macro(tn, 'auprc'):.3f}")
w = RUNS['teacher']['states'][0]
print('parametri del gate (fold 0):', {k.split('.', 1)[1]: round(float(v), 3) for k, v in w.items() if k.startswith('gate.')},
      '→ devono essersi MOSSI da (w=1, b=0): nel progetto di riferimento restavano a 1.0 perché mai addestrati')

## Sub-Task 5 · Calibrazione e quote misurate

In [ ]:
# ---------- Sub-Task 5: calibrazione (Temperature Scaling) e confronto con le quote MISURATE ------------------------------------------------
# T_org e T_neu sono stati stimati su validation in ogni fold (LBFGS) e applicati ai logits del fold di test. Qui si misura l'effetto.
# Attenzione al significato: frac_cyto/frac_nuc sono la quota SOLUBILE misurata per frazionamento in HEK293T/HeLa (Hein 2024, Itzhak 2016), mentre la
# probabilità predetta è la quota di massa del compartimento "Cytoplasm" (che include citoscheletro). Sono grandezze vicine, non identiche:
# per questo si guardano correlazione di rango e reliability diagram, non l'uguaglianza esatta.
from scipy.stats import spearmanr
res = RUNS['teacher']
has_frac = ~np.isnan(prot['frac_cyto'].to_numpy()) & ~is_test
fig, ax = plt.subplots(2, 2, figsize=(11, 9))
print('Temperature per fold (T_org, T_neu):', np.round(res['T'], 2).tolist(), '  (T > 1 = rete troppo sicura, T < 1 = troppo timida)')
for j, (cname, fcol) in enumerate([('Cytoplasm', 'frac_cyto'), ('Nucleus', 'frac_nuc')]):
    c = ORG.index(cname); meas = prot[fcol].to_numpy()
    for i, (key, lab) in enumerate([('po_raw', 'prima (softmax grezza)'), ('po', 'dopo Temperature Scaling')]):
        pred = res['R']['full'][key][:, c]
        m = has_frac & ~np.isnan(meas)
        e, rho = ece_continuous(pred[m], np.clip(meas[m], 0, 1)), spearmanr(pred[m], meas[m]).correlation
        a = ax[j, i]; a.scatter(pred[m], meas[m], s=3, alpha=.25, color='#888')
        bins = np.linspace(0, 1, 11); b = np.digitize(pred[m], bins[1:-1])
        a.plot([(bins[x] + bins[x + 1]) / 2 for x in range(10) if (b == x).any()], [meas[m][b == x].mean() for x in range(10) if (b == x).any()], 'o-', color='#D85A30', label='media per bin')
        a.plot([0, 1], [0, 1], 'k--', lw=.8); a.set_title(f'{cname} · {lab}\nECE={e:.3f} · Spearman={rho:.2f} · n={int(m.sum())}', fontsize=9)
        a.set_xlabel('probabilità predetta'); a.set_ylabel('frazione misurata'); a.legend(fontsize=7)
        print(f'{cname:10s} {lab:28s} ECE_continuo={e:.3f}  Spearman={rho:.2f}')
plt.tight_layout(); plt.savefig(FIG / 'calibrazione_quote_misurate.png', dpi=130); plt.show()

# ECE di classificazione contro REF, per classe affidabile: prima vs dopo
tab = []
for c, n in enumerate(ALL):
    if not RELIABLE[c]:
        continue
    k_raw, k_cal = ('po_raw', 'pn_raw'), ('po', 'pn')
    get = lambda keys: res['R']['full'][keys[0]][:, c] if c < NO else res['R']['full'][keys[1]][:, c - NO]
    ok = ~np.isnan(REF[:, c])
    tab.append((n, ece_binary(get(k_raw)[ok], REF[ok, c]), ece_binary(get(k_cal)[ok], REF[ok, c])))
ece_t = pd.DataFrame(tab, columns=['classe', 'ECE_prima', 'ECE_dopo']).set_index('classe')
print('\nECE contro REF (classi affidabili):\n', ece_t.round(3).to_string(), f"\nmedia: prima {ece_t['ECE_prima'].mean():.3f} → dopo {ece_t['ECE_dopo'].mean():.3f}")

## Sub-Task 6a · Geometria dello spazio latente con SupCon

In [ ]:
# ---------- Sub-Task 6a: la geometria di repr_256 con e senza SupCon -----------------------------------------------------------------------
try:
    import umap
    emb2d = lambda X: umap.UMAP(n_components=2, n_neighbors=25, min_dist=0.2, random_state=SEED).fit_transform(X)
    method = 'UMAP'
except Exception:
    emb2d = lambda X: PCA(2, random_state=SEED).fit_transform(X)
    method = 'PCA (umap-learn non installato: pip install umap-learn)'
if 'no_supcon' in RUNS:
    rare_cls = [j for j in range(NN) if 0 < n_pos[NO + j] < HYPER['supcon_rare_max']]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
    sel = np.flatnonzero(is_train & (np.abs(RUNS['teacher']['repr']).sum(1) > 0))
    sel = np.random.RandomState(SEED).choice(sel, min(len(sel), 4000), replace=False)
    pal = ['#D85A30', '#1D9E75', '#7F77DD', '#BA7517', '#378ADD']
    for a, key, title in [(ax[0], 'no_supcon', 'senza SupCon (β=0)'), (ax[1], 'teacher', f"con SupCon (β={HYPER['beta_supcon']})")]:
        Z = emb2d(RUNS[key]['repr'][sel]); a.scatter(Z[:, 0], Z[:, 1], s=3, c='#cccccc')
        for j in rare_cls:
            m = REF[sel, NO + j] == 1
            a.scatter(Z[m, 0], Z[m, 1], s=14, c=pal[j % 5], label=f'{NEU[j]} (n={int(m.sum())})')
        a.set_title(f'{title} · {method}', fontsize=10); a.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG / 'supcon_repr256.png', dpi=130); plt.show()
    print(summary(RUNS['teacher']), '\n', summary(RUNS['no_supcon']), '\n(prima riga: con SupCon · seconda: senza)')

## Sub-Task 3 · Indice di Leakage Semantico

In [ ]:
# ---------- Sub-Task 3 · Indice di Leakage Semantico I_leak = MI(E_text, Y) / MI(E_text_full, Y) ------------------------------------------
# MI stimata con k-NN (mutual_info_classif) sulle prime 32 componenti PCA degli embedding BioLinkBERT contro l'etichetta primaria (organello dominante).
# Le componenti PCA sono decorrelate, quindi la somma delle MI per componente è un indice RELATIVO ragionevole (non una MI assoluta).
# COME LEGGERLO — il piano fissa la soglia 0,35 ("card validata se I_leak < 0,35"). Attenzione: I_leak misura quanta informazione sul bersaglio RESTA nel testo,
# non quanta sia leakage. La funzione di una proteina ("fattore di trascrizione") è informativa legittimamente, quindi una card onesta può avere I_leak ben
# sopra 0,35. La lettura corretta è comparativa: protein deve stare ben sotto 1,0 (la localizzazione dichiarata è stata tolta) e degraded sotto protein
# (anche i proxy di famiglia sono stati tolti). L'esito PASS/FAIL sulla soglia viene stampato, ma la prova decisiva è la tabella di ablazione.
rows_mi = np.flatnonzero(M_org & is_train)
rows_mi = np.random.RandomState(SEED).choice(rows_mi, min(len(rows_mi), 6000), replace=False)
y_prim = Y_org[rows_mi].argmax(1)
mi = {}
for m in CARD_MODES:
    Zp = PCA(32, random_state=SEED).fit_transform(E_text[m][rows_mi].astype(np.float32))
    mi[m] = float(mutual_info_classif(Zp, y_prim, discrete_features=False, n_neighbors=5, random_state=SEED).sum())
ileak = pd.Series({'full': 1.0, 'protein': mi['protein'] / mi['full'], 'degraded': mi['degraded'] / mi['full']})
print('MI (nat, somma su 32 componenti):', {k: round(v, 3) for k, v in mi.items()})
print('I_leak:', ileak.round(3).to_dict())
print(f"soglia del piano (card_protein < 0,35): {'PASS' if ileak['protein'] < 0.35 else 'NON superata'} · degraded/full = {ileak['degraded']:.2f}")

## Sub-Task 6b · Student per proteine orfane

In [ ]:
# ---------- Sub-Task 6b: Teacher → Student (solo sequenza + AlphaFold), Dual-Head Feature & Logit Distillation ------------------------------
# L_KD = T²·KL(softmax(z_org_T/T) ‖ softmax(z_org_S/T)) + BCE(σ(z_neu_T/T), σ(z_neu_S/T)) + β·MSE(repr_T, repr_S).
# Modifiche rispetto al piano: (1) aggiunto λ·(loss dura) sulle etichette vere: un Student puro-distillazione eredita anche gli errori del Teacher, e il
# Teacher è sovra-sicuro proprio sulle proteine su cui è stato addestrato; (2) la distillazione usa anche proteine del cervello senza etichetta
# (mai quelle del test HPA); (3) il Teacher usato è quello del fold, congelato, con la card ripulita.
RUNS['student'] = run_cv('student', kind='student', teacher_states=RUNS['teacher']['states'], teacher_mode='protein', keep_states=True)
low = card_n_annot < LOW_ANNOT_WORDS
cmp = {}
for nm, P in [('DeepLoc-style (ESM-2 → MLP)', P_dl), ('Teacher (con card)', None), ('Teacher card-free', None), ('Student (seq+AlphaFold)', None)]:
    if nm.startswith('Teacher (') or nm.startswith('Teacher card'):
        v = 'full' if 'con card' in nm else 'free'; r = RUNS['teacher']['R'][v]
    elif nm.startswith('Student'):
        r = RUNS['student']['R']['full']
    else:
        r = None
    P = P if r is None else np.concatenate([r['po'], r['pn']], 1)
    for scen, rows in [('Full', None), ('Low-Annotation', low)]:
        tA, tB = table(P[:, :NO], REF[:, :NO], ORG, rows), table(P[:, NO:], REF[:, NO:], NEU, rows)
        cmp[(nm, scen)] = dict(AUROC_A=macro(tA), AUPRC_A=macro(tA, 'auprc'), AUROC_B=macro(tB), AUPRC_B=macro(tB, 'auprc'))
cmp = pd.DataFrame(cmp).T.round(3)
print(cmp.to_string())
tf = cmp.loc[('Teacher (con card)', 'Full')]; sf = cmp.loc[('Student (seq+AlphaFold)', 'Full')]
print(f"\nGap Teacher→Student (valore aggiunto della card_protein), AUROC_A: {tf['AUROC_A'] - sf['AUROC_A']:+.3f} · AUPRC_B: {tf['AUPRC_B'] - sf['AUPRC_B']:+.3f}")
dl_ = cmp.loc[('DeepLoc-style (ESM-2 → MLP)', 'Full')]
print(f"Student vs baseline solo-sequenza: AUROC_A {sf['AUROC_A'] - dl_['AUROC_A']:+.3f} · AUPRC_B {sf['AUPRC_B'] - dl_['AUPRC_B']:+.3f}")

## Curve ROC finali: rete neurale vs baseline vs regressione logistica
Pannello 1-2: previsioni out-of-fold con fold per omologia. Pannello 3: test HPA indipendente (proteine mai viste, solo organelli visibili al microscopio). Tutte le curve usano gli stessi fold e lo stesso riferimento REF.

In [ ]:
# ---------- CURVE ROC FINALI: rete neurale finale vs baseline vs regressione logistica (e DeepLoc-style, Student) ------------------------------
MODELS_CMP = {'Baseline (prevalenza)': P_prev, 'Regressione logistica': P_lr, 'DeepLoc-style (ESM-2→MLP)': P_dl,
              'NeuroLoc Student (seq+AF)': np.concatenate([RUNS['student']['R']['full']['po'], RUNS['student']['R']['full']['pn']], 1),
              'NeuroLoc Teacher (rete finale)': np.concatenate([RUNS['teacher']['R']['full']['po'], RUNS['teacher']['R']['full']['pn']], 1)}
STYLE = {'Baseline (prevalenza)': ('#999999', ':', 1.2), 'Regressione logistica': ('#378ADD', '--', 1.6), 'DeepLoc-style (ESM-2→MLP)': ('#BA7517', '--', 1.6),
         'NeuroLoc Student (seq+AF)': ('#1D9E75', '-', 1.8), 'NeuroLoc Teacher (rete finale)': ('#D85A30', '-', 2.4)}
FPR = np.linspace(0, 1, 201)

def macro_roc(P, Y, cls):
    tprs, aucs = [], []
    for c in cls:
        ok = ~np.isnan(Y[:, c])
        y, p = Y[ok, c], P[ok, c]
        if (y == 1).sum() < MIN_EVAL_POS or (y == 0).sum() < MIN_EVAL_POS:
            continue
        f, t, _ = roc_curve(y, p + 1e-9 * np.random.RandomState(0).rand(len(p)))     # rumore minuscolo: la prevalenza costante dà una diagonale pulita
        tprs.append(np.interp(FPR, f, t)); aucs.append(roc_auc_score(y, p))
    return np.mean(tprs, 0), np.mean(aucs), len(aucs)

def panel(a, Y, cls, title, rows=None):
    for nm, P in MODELS_CMP.items():
        t, auc, n = macro_roc(P if rows is None else np.where(rows[:, None], P, np.nan), Y if rows is None else np.where(rows[:, None], Y, np.nan), cls)
        col, ls, lw = STYLE[nm]
        a.plot(FPR, t, color=col, ls=ls, lw=lw, label=f'{nm}  AUC={auc:.3f}')
    a.plot([0, 1], [0, 1], color='#ccc', lw=.6); a.set_title(f'{title} · {n} classi', fontsize=10)
    a.set_xlabel('falsi positivi'); a.set_ylabel('veri positivi'); a.legend(fontsize=7, loc='lower right')

fig, ax = plt.subplots(1, 3, figsize=(18, 5.6))
panel(ax[0], REF, range(NO), 'Asse A · organelli · OOF omologia-separata')
panel(ax[1], REF, range(NO, NO + NN), 'Asse B · domini neuronali · OOF')
Yt = np.full((N, len(ALL)), np.nan); Yt[:, :NO] = Y_test
panel(ax[2], Yt, range(NO), 'Test indipendente HPA · organelli', rows=is_test)
plt.tight_layout(); plt.savefig(FIG / 'roc_confronto_modelli.png', dpi=140); plt.show()

# per classe: 14 pannelli
fig, axs = plt.subplots(4, 4, figsize=(18, 16)); axs = axs.ravel()
for c, n in enumerate(ALL):
    ok = ~np.isnan(REF[:, c]); a = axs[c]
    if not RELIABLE[c]:
        a.text(.5, .5, f'{n}\nn_pos={int(n_pos[c])} < {MIN_EVAL_POS}\nnon valutabile', ha='center', va='center'); a.axis('off'); continue
    for nm, P in MODELS_CMP.items():
        f, t, _ = roc_curve(REF[ok, c], P[ok, c] + 1e-9 * np.random.RandomState(0).rand(ok.sum())); col, ls, lw = STYLE[nm]
        a.plot(f, t, color=col, ls=ls, lw=lw, label=f'{nm.split(" (")[0]} {roc_auc_score(REF[ok, c], P[ok, c]):.2f}')
    a.plot([0, 1], [0, 1], color='#ccc', lw=.6); a.set_title(f'{n} (n+={int(n_pos[c])})', fontsize=9); a.legend(fontsize=6, loc='lower right')
for a in axs[len(ALL):]:
    a.axis('off')
plt.tight_layout(); plt.savefig(FIG / 'roc_per_classe.png', dpi=120); plt.show()

# tabella finale con intervalli di confidenza bootstrap (ricampionamento delle proteine, appaiato tra modelli)
rng = np.random.RandomState(SEED); valid = np.flatnonzero(is_train & (fold >= 0)); B = 200
def macro_auc(P, idx, cls):
    a = []
    for c in cls:
        y = REF[idx, c]; ok = ~np.isnan(y)
        if RELIABLE[c] and 0 < (y[ok] == 1).sum() < ok.sum():
            a.append(roc_auc_score(y[ok], P[idx][ok, c]))
    return np.mean(a) if a else np.nan
boot = {nm: {'A': [], 'B': []} for nm in MODELS_CMP}
for _ in range(B):
    idx = rng.choice(valid, len(valid))
    for nm, P in MODELS_CMP.items():
        boot[nm]['A'].append(macro_auc(P, idx, range(NO))); boot[nm]['B'].append(macro_auc(P, idx, range(NO, NO + NN)))
final = []
for nm, P in MODELS_CMP.items():
    r = dict(modello=nm)
    for ax_, cls in [('A', range(NO)), ('B', range(NO, NO + NN))]:
        pt = macro_auc(P, valid, cls); lo_, hi_ = np.nanpercentile(boot[nm][ax_], [2.5, 97.5])
        r[f'AUROC_{ax_}'] = f'{pt:.3f} [{lo_:.3f}–{hi_:.3f}]'
        r[f'AUPRC_{ax_}'] = round(macro(table(P[:, cls.start:cls.stop], REF[:, cls.start:cls.stop], ALL[cls.start:cls.stop]), 'auprc'), 3)
    final.append(r)
final = pd.DataFrame(final).set_index('modello')
print(final.to_string())
dA = np.array(boot['NeuroLoc Teacher (rete finale)']['A']) - np.array(boot['Regressione logistica']['A'])
dB = np.array(boot['NeuroLoc Teacher (rete finale)']['B']) - np.array(boot['Regressione logistica']['B'])
print(f"\nTeacher − logistica (bootstrap appaiato): ΔAUROC Asse A {np.nanmean(dA):+.3f} [{np.nanpercentile(dA, 2.5):+.3f}, {np.nanpercentile(dA, 97.5):+.3f}] · "
      f"Asse B {np.nanmean(dB):+.3f} [{np.nanpercentile(dB, 2.5):+.3f}, {np.nanpercentile(dB, 97.5):+.3f}]  (un intervallo che include 0 = differenza non dimostrata)")
final.to_csv(RESULTS / 'roc_confronto_02.csv')

## Output per i ricercatori e file per lo script 03

In [ ]:
# ---------- Output per i ricercatori: percentuali per compartimento + probabilità dei domini neuronali + spiegazione ---------------------------
RT, RS = RUNS['teacher']['R']['full'], RUNS['student']['R']['full']
pct = pd.DataFrame(RT['po'] * 100, columns=['pct_' + c for c in ORG]).round(1)        # Asse A: somma 100 %
pneu = pd.DataFrame(RT['pn'], columns=['p_' + c for c in NEU]).round(3)               # Asse B: probabilità indipendenti
out = pd.concat([prot[['uniprot_id', 'gene', 'protein_name', 'role', 'card_n_annot']], pct, pneu], axis=1)
out['top_organello'] = np.array(ORG)[RT['po'].argmax(1)]
out['incertezza_entropia'] = (-(RT['po'] * np.log(RT['po'] + 1e-9)).sum(1) / math.log(NO)).round(3)    # 0 = una sola localizzazione netta, 1 = distribuzione piatta
out['top_organello_student'] = np.array(ORG)[RS['po'].argmax(1)]                       # per proteine orfane: usa questa colonna
for j, nme in enumerate(['sequenza_intera', 'N_term', 'centro', 'C_term']):
    out['card_interroga_' + nme] = RUNS['teacher']['att'][:, j].round(3)                # quale pezzo della proteina la card "guarda" (cross-attention)
out['origine'] = np.where(fold >= 0, 'out-of-fold', np.where(is_test, 'test_HPA (media 5 fold)', 'non etichettata (media 5 fold)'))
out.to_csv(RESULTS / 'predizioni_cervello.csv', index=False)
print(out.head(8).to_string()); print(f"\nsalvato results/predizioni_cervello.csv ({len(out):,} proteine)")
assert np.allclose(pct.sum(1), 100, atol=0.6), 'le percentuali dell\'Asse A devono sommare 100'

# ---------- salvataggi per lo script 03: modelli, predizioni OOF, fold, soglie ----------------------------------------------------------------
torch.save(dict(teacher_states=RUNS['teacher']['states'], student_states=RUNS['student']['states'], HYPER=HYPER, ESM_DIM=ESM_DIM, BERT_DIM=BERT_DIM,
                STRUCT_DIM=STRUCT_DIM, mu=mu, sd=sd, T=RUNS['teacher']['T'], classes=ALL), MODELS / 'mm_can_subcell_best.pt')
np.savez_compressed(RESULTS / 'oof_02.npz', REF=REF, Y_test=Y_test, fold=fold, groups=groups, thr=RUNS['teacher']['thr'], classes=np.array(ALL),
                    P_teacher=np.concatenate([RT['po'], RT['pn']], 1), P_student=np.concatenate([RS['po'], RS['pn']], 1), P_lr=P_lr, P_dl=P_dl, P_prev=P_prev,
                    ECE_org_raw=np.array([ece_binary(RT['po_raw'][:, c][~np.isnan(REF[:, c])], REF[~np.isnan(REF[:, c]), c]) for c in range(NO)]))
np.savez_compressed(RESULTS / 'features_02.npz', F_struct=F_struct, af_missing=af_missing, card_n_annot=card_n_annot, Y_org=Y_org, M_org=M_org,
                    Y_neu=Y_neu, M_neu=M_neu, is_train=is_train, is_test=is_test)
print('salvati: models/mm_can_subcell_best.pt · results/oof_02.npz · results/features_02.npz')

def scheda(q, top=5):
    """Scheda di una proteina: percentuali per organello, domini neuronali e cosa ha guidato la predizione. q = simbolo genico o UniProt ID."""
    r = out[(out['gene'].str.upper() == q.upper()) | (out['uniprot_id'] == q)]
    if r.empty:
        print('non trovata nella coorte cerebrale'); return
    r = r.iloc[0]; i = r.name
    print(f"{r['gene']} ({r['uniprot_id']}) · {r['protein_name']}\norigine della predizione: {r['origine']} · parole di annotazione: {int(r['card_n_annot'])}"
          f"{'  ← poco annotata: preferire il Student' if r['card_n_annot'] < LOW_ANNOT_WORDS else ''}")
    print('\nASSE A — dove si trova (quota %, somma 100):')
    for c, v in sorted(zip(ORG, pct.iloc[i]), key=lambda x: -x[1])[:top]:
        print(f'  {c:24s} {v:5.1f} % ' + '█' * int(v / 4))
    print('ASSE B — probabilità di presenza nel dominio neuronale:')
    for c in NEU:
        print(f'  {c:14s} {pneu.iloc[i]["p_" + c]:.2f} ' + '█' * int(pneu.iloc[i]['p_' + c] * 25))
    att = {n: r['card_interroga_' + n] for n in ['sequenza_intera', 'N_term', 'centro', 'C_term']}
    print('la card ha interrogato soprattutto:', max(att, key=att.get), att, f'\nincertezza (entropia normalizzata): {r["incertezza_entropia"]}')
scheda(prot.loc[prot['has_label'], 'gene'].iloc[0])